# Lab 7 — 정말 정규분포인가? (QQ plot)

**확률통계 · Topic 7 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. **QQ plot** 읽는 법을 배운다 — 분포 가정을 검증하는 표준 도구.
2. 히스토그램으로는 안 보이던 **꼬리의 차이**를 QQ plot으로 잡아낸다.
3. 주가 수익률이 정규분포가 **아니라는 것**을 직접 확인한다.

⏱ **예상 소요 시간: 35분**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. QQ plot의 원리

아이디어는 간단하다.

1. 내 데이터를 **크기순으로 정렬**한다.
2. "정규분포였다면 이 순위에 어떤 값이 있어야 하는가"를 계산한다.
3. 둘을 **점으로 찍는다.** 정규분포가 맞다면 **직선** 위에 놓인다.

### 실습 1 — QQ plot 함수 만들기

In [ ]:
def qq_plot(data, ax, title=""):
    data = np.sort(data)
    n = len(data)
    q = (np.arange(1, n + 1) - 0.5) / n

    # TODO 1: 같은 평균/표준편차의 정규분포라면 나왔을 값을 구하세요
    #         힌트: stats.norm.ppf(q, data.mean(), data.std())
    theory = np.zeros(n)

    ax.plot(theory, data, ".", ms=3)
    lo, hi = min(theory[0], data[0]), max(theory[-1], data[-1])
    ax.plot([lo, hi], [lo, hi], color="red", lw=2)
    ax.set_xlabel("Theoretical (normal)")
    ax.set_ylabel("Observed")
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(4.5, 4.5))
qq_plot(rng.normal(0, 1, 3000), ax, "Really normal")
plt.show()

> 점들이 **빨간 직선 위에** 놓였다면 성공이다. 이것이 "정규분포가 맞다"의 모습이다.

## Part 2. 세 데이터를 검증하자

- **시험 점수** — 여러 문항 점수의 합
- **소득** — 오른쪽으로 길게 늘어진 것으로 알려져 있다
- **주가 일간 수익률** — ?

### 실습 2 — 세 데이터 만들기 (합성)

In [ ]:
N = 3000

scores = rng.normal(72, 12, N).clip(0, 100)          # 시험 점수
income = rng.lognormal(mean=8.0, sigma=0.7, size=N)  # 소득 (로그정규)
returns = stats.t.rvs(df=3, size=N, random_state=20260302)
returns = returns / returns.std() * 1.2               # 일간 수익률(%)

for name, d in [("시험점수", scores), ("소득", income), ("수익률", returns)]:
    print(f"{name:>6}  평균 {d.mean():10.2f}  표준편차 {d.std():9.2f}  "
          f"최소 {d.min():9.2f}  최대 {d.max():10.2f}")

### 실습 3 — 히스토그램 먼저 (여기서는 잘 안 보인다)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, (name, d) in zip(axes, [("Exam scores", scores), ("Income", income),
                                ("Daily returns", returns)]):
    ax.hist(d, bins=60, density=True, alpha=0.8)
    xs = np.linspace(d.min(), d.max(), 300)
    # TODO 2: 같은 평균/표준편차의 정규분포 PDF를 겹쳐 그리세요
    #         힌트: stats.norm.pdf(xs, d.mean(), d.std())

    ax.set_title(name)
plt.tight_layout()
plt.show()

소득은 한눈에 봐도 치우쳐 있다. 그런데 **수익률은 정규분포와 꽤 비슷해 보인다.**

### 실습 4 — 같은 데이터를 QQ plot으로

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
# TODO 3: 세 데이터의 QQ plot을 각각 그리세요
#         힌트: qq_plot(scores, axes[0], "Exam scores")

plt.tight_layout()
plt.show()

🤔 **히스토그램에서는 비슷해 보이던 수익률이, QQ plot에서는 양끝이 확 휘어 있다.**

- **직선** → 정규분포에 잘 맞는다
- **양끝이 위/아래로 휜다** → **꼬리가 두껍다** (극단값이 정규분포 예측보다 훨씬 자주)
- **한쪽으로 굽는다** → 비대칭 (소득이 이 모양)

## Part 3. 꼬리를 숫자로

"꼬리가 두껍다"를 눈이 아니라 **숫자로** 확인하자.

### 실습 5 — 몇 시그마를 넘는가

In [ ]:
def tail_report(data, name):
    # TODO 4: 데이터를 표준화하세요.  힌트: (data - data.mean()) / data.std()
    z = np.zeros(len(data))
    print(f"[{name}]")
    for k in [2, 3, 4, 5]:
        actual = float((np.abs(z) > k).mean())
        theory = float(2 * stats.norm.sf(k))
        print(f"  |z| > {k}:  실제 {actual:8.5f}   정규분포라면 {theory:8.5f}")


tail_report(scores, "시험 점수")
tail_report(returns, "일간 수익률")

### 실습 6 — "백만 년에 한 번" 계산해보기

정규분포를 가정하면 $5\sigma$ 사건은 얼마나 드문가?

In [ ]:
p5 = 2 * stats.norm.sf(5)
print(f"정규분포에서 |z| > 5 인 확률: {p5:.3e}")
print(f"거래일 기준 몇 년에 한 번? 약 {1 / p5 / 250:,.0f}년에 한 번")

z_ret = (returns - returns.mean()) / returns.std()
# TODO 5: |z| > 5 인 날이 며칠인지 세세요.  힌트: int((np.abs(z_ret) > 5).sum())
n_big = 0
print(f"\n그런데 우리 데이터 {len(returns)}일 중 |z| > 5 인 날: {n_big}일")
print(f"정규분포라면 기대되는 날 수: {p5 * len(returns):.4f}일")

😱 **정규분포를 믿었다면 "수천 년에 한 번"이라고 계산했을 사건이
3,000일 데이터 안에서 여러 번 일어났다.**

> 2008년 금융위기 때 여러 위험 관리 모형이 정확히 이 실수를 했다.
> **모형이 틀린 것이 아니라, 정규분포라는 가정이 틀렸던 것이다.**

---

## 마무리 — 자가 점검

- [ ] QQ plot을 그리고 읽을 수 있다
- [ ] 히스토그램만으로는 꼬리를 판단할 수 없다는 것을 확인했다
- [ ] 표준화해서 몇 시그마를 넘는지 셀 수 있다
- [ ] 정규분포 가정이 위험한 경우를 하나 이상 설명할 수 있다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)

### 📌 다음 주는 **중간고사(1차시) + 미니 프로젝트 1 발표(2차시)** 입니다